# 03 — Baseline Model and Initial Model Training

## Objective

The objective of this notebook is to establish a baseline and compare different machine learning models for distinguishing between computer-generated (CG) and original (OR) consumer reviews.

We experiment with:

1. Dummy Classifier
2. Logistic Regression with unigram TF-IDF
3. Logistic Regression with unigram and bigram TF-IDF
4. Linear Support Vector Machine with unigram and bigram TF-IDF

All models are evaluated using stratified 5-fold cross-validation on the training dataset.

The test dataset is kept separate and will only be used for final evaluation after model selection.

In [44]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

## 1. Loading the Processed Dataset

We load the training dataset generated during preprocessing.

The training dataset contains the review text and its encoded target label:

- 0 = Original Review (OR)
- 1 = Computer Generated Review (CG)

The test dataset is deliberately not loaded in this notebook because we will use it only for final evaluation.

In [45]:
# Load the processed training dataset
train_df = pd.read_csv("../data/processed/train.csv")

# Separate features and target
X_train = train_df["text"].astype(str)
y_train = train_df["target"].astype(int)

In [46]:
print("Training set shape:", X_train.shape)

print("\nLabel distribution:")
print(y_train.value_counts().sort_index())

print("\nMissing values in review text:", X_train.isnull().sum())

print("\nMissing values in target:", y_train.isnull().sum())

print("\nDuplicate reviews:", X_train.duplicated().sum())

Training set shape: (32328,)

Label distribution:
target
0    16172
1    16156
Name: count, dtype: int64

Missing values in review text: 0

Missing values in target: 0

Duplicate reviews: 0


## 2. Evaluation Methodology

We use stratified 5-fold cross-validation to compare the models.

In each iteration:
- Four folds are used for training.
- One fold is used for validation.
- The process repeats until every fold has been used for validation.

Stratification ensures that the proportion of CG and OR reviews remains approximately consistent across folds.

We evaluate the models using:

1. Accuracy
2. Macro F1-score
3. Balanced Accuracy
4. ROC-AUC

All TF-IDF transformations are included inside the model pipelines. This ensures that the vocabulary and IDF weights are learned separately within each training fold, preventing information leakage from validation folds.

In [47]:
# Define stratified 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc"
}

In [48]:
def display_cv_results(results, model_name):
    print(f"\n{model_name}")
    print("-" * 50)

    for metric in scoring:
        scores = results[f"test_{metric}"]

        print(
            f"{metric}: "
            f"{scores.mean():.4f} ± {scores.std():.4f}"
        )

## 3. Dummy Classifier — Baseline

We first establish a baseline using a Dummy Classifier.

The classifier uses the `most_frequent` strategy, meaning it always predicts the majority class in the training data.

It does not learn any meaningful patterns from the review text.

This provides a reference point against which the performance of our actual machine learning models can be compared.

In [49]:
# Define the Dummy Classifier
dummy_model = DummyClassifier(
    strategy="most_frequent"
)

# Perform 5-fold cross-validation
dummy_cv = cross_validate(
    dummy_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Display results
display_cv_results(dummy_cv, "Dummy Classifier")


Dummy Classifier
--------------------------------------------------
accuracy: 0.5002 ± 0.0001
macro_f1: 0.3334 ± 0.0000
balanced_accuracy: 0.5000 ± 0.0000
roc_auc: 0.5000 ± 0.0000


## 4. Logistic Regression with Unigram TF-IDF

We now train our first actual machine learning model.

TF-IDF converts review text into numerical feature vectors.

We initially use unigram features, which represent individual words.

Logistic Regression learns the relationship between these word features and the two target classes.

In [50]:
# Create a pipeline with TF-IDF and Logistic Regression
lr_unigram = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=2,
        max_df=0.95
    )),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Perform cross-validation
lr_unigram_cv = cross_validate(
    lr_unigram,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Display results
display_cv_results(
    lr_unigram_cv,
    "Logistic Regression — Unigrams"
)


Logistic Regression — Unigrams
--------------------------------------------------
accuracy: 0.8993 ± 0.0035
macro_f1: 0.8993 ± 0.0035
balanced_accuracy: 0.8993 ± 0.0035
roc_auc: 0.9636 ± 0.0014


## 5. Logistic Regression with Unigrams and Bigrams

We now extend the TF-IDF representation to include both unigrams and bigrams.

Unigrams represent individual words, while bigrams represent consecutive pairs of words.

Including bigrams allows the model to capture some word-order and phrase-level information that individual word features cannot directly represent.

We compare its performance with the unigram-only model to investigate whether bigram features improve classification.

In [51]:
# TF-IDF with unigrams and bigrams + Logistic Regression
lr_bigram = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Perform cross-validation
lr_bigram_cv = cross_validate(
    lr_bigram,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Display results
display_cv_results(
    lr_bigram_cv,
    "Logistic Regression — Unigrams + Bigrams"
)


Logistic Regression — Unigrams + Bigrams
--------------------------------------------------
accuracy: 0.9256 ± 0.0024
macro_f1: 0.9255 ± 0.0024
balanced_accuracy: 0.9256 ± 0.0024
roc_auc: 0.9785 ± 0.0007


## 6. Linear Support Vector Machine with Unigrams and Bigrams

We now experiment with Linear Support Vector Machine (Linear SVM).

Linear SVM attempts to find a decision boundary that separates the two classes while maximizing the margin between them.

It is commonly used for text classification because it performs well with high-dimensional and sparse TF-IDF feature representations.

We use the same unigram and bigram TF-IDF configuration as the previous Logistic Regression model to make the comparison more meaningful.

In [52]:
# TF-IDF with unigrams and bigrams + Linear SVM
svm_bigram = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )),
    ("classifier", LinearSVC(
        max_iter=5000,
        random_state=42
    ))
])

# Perform cross-validation
svm_bigram_cv = cross_validate(
    svm_bigram,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Display results
display_cv_results(
    svm_bigram_cv,
    "Linear SVM — Unigrams + Bigrams"
)


Linear SVM — Unigrams + Bigrams
--------------------------------------------------
accuracy: 0.9439 ± 0.0013
macro_f1: 0.9439 ± 0.0013
balanced_accuracy: 0.9439 ± 0.0013
roc_auc: 0.9867 ± 0.0003


## 7. Initial Model Comparison

We consolidate the cross-validation results of all four model configurations.

The comparison uses the mean and standard deviation of each evaluation metric across the five folds.

This allows us to compare the models systematically and identify the strongest initial candidate for further experimentation.

In [53]:
# Store the cross-validation results of all models
model_results = {
    "Dummy Classifier": dummy_cv,
    "Logistic Regression (Unigrams)": lr_unigram_cv,
    "Logistic Regression (Unigrams + Bigrams)": lr_bigram_cv,
    "Linear SVM (Unigrams + Bigrams)": svm_bigram_cv
}

# Create the comparison table
comparison = []

for model_name, results in model_results.items():

    row = {"Model": model_name}

    for metric in scoring:
        scores = results[f"test_{metric}"]

        row[f"{metric}_mean"] = scores.mean()
        row[f"{metric}_std"] = scores.std()

    comparison.append(row)

comparison_df = pd.DataFrame(comparison)

# Display the comparison table
comparison_df

,Model,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std
0,Dummy Classifier,0.500247,0.000058,0.333443,0.000026,0.500000,0.000000,0.500000,0.000000
1,Logistic Regression (Unigrams),0.899344,0.003452,0.899323,0.003454,0.899337,0.003453,0.963559,0.001408
2,Logistic Regression (Unigrams + Bigrams),0.925575,0.002407,0.925543,0.002412,0.925565,0.002410,0.978464,0.000675
3,Linear SVM (Unigrams + Bigrams),0.943949,0.001281,0.943946,0.001281,0.943945,0.001281,0.986693,0.000330


In [54]:
# Save mean and standard deviation results
comparison_df.to_csv(
    "../results/metrics/model_comparison_cv.csv",
    index=False
)

# Store individual fold scores
fold_results = []

for model_name, results in model_results.items():

    for fold in range(5):

        row = {
            "Model": model_name,
            "Fold": fold + 1
        }

        for metric in scoring:
            row[metric] = results[f"test_{metric}"][fold]

        fold_results.append(row)

fold_comparison_df = pd.DataFrame(fold_results)

# Save individual fold results
fold_comparison_df.to_csv(
    "../results/metrics/model_comparison_cv_folds.csv",
    index=False
)

print("Cross-validation results saved successfully.")

Cross-validation results saved successfully.


## 8. Observations and Conclusion

The initial experiments show that all three actual machine learning configurations perform substantially better than the Dummy Classifier baseline.

Logistic Regression with unigram TF-IDF provides a strong initial result.

Adding bigram features improves the performance of Logistic Regression, suggesting that combinations of consecutive words contain additional useful information for distinguishing between the two dataset labels.

Linear SVM with unigram and bigram TF-IDF achieves the strongest performance among the initial configurations, with approximately:

- Accuracy: 94.39%
- Macro F1: 94.39%
- Balanced Accuracy: 94.39%
- ROC-AUC: 98.67%

Based on these initial cross-validation results, Linear SVM with unigram and bigram TF-IDF is selected as the leading candidate for further experimentation.

Further experiments will investigate whether changes to the feature extraction settings can improve performance.

These results measure the model's ability to reproduce the CG and OR labels in the selected dataset. They should not be interpreted as proof of reliable AI-generated review detection across arbitrary real-world sources.